# Phase 5: Composite Manipulation Scoring

Calculate weighted manipulation scores from all forensic vectors. Five sub-scores (oscillation, fabrication, causality, Polymarket, and Gemini intent) are combined into a single composite score with LOW / ELEVATED / HIGH verdicts.

## Step 1: Load All Data Sources

Load the master timeline, Gemini classifications, and Polymarket suspicion scores.

In [2]:
import pandas as pd
import numpy as np
import os

PROCESSED = '../data/processed'

In [ ]:
master = pd.read_csv(os.path.join(PROCESSED, 'master.csv'))
master['date'] = pd.to_datetime(master['date'])

# Load Gemini classifications
gemini_path = os.path.join(PROCESSED, 'gemini_classifications.csv')
if os.path.exists(gemini_path):
    gemini = pd.read_csv(gemini_path)
    print(f"Gemini classifications: {len(gemini)}")
else:
    # Fall back to partial
    gemini_path = os.path.join(PROCESSED, 'gemini_partial.csv')
    if os.path.exists(gemini_path):
        gemini = pd.read_csv(gemini_path)
        print(f"Gemini partial classifications: {len(gemini)}")
    else:
        gemini = pd.DataFrame()
        print("No Gemini data available")

# Load Polymarket
pm_path = os.path.join(PROCESSED, 'polymarket_suspicion.csv')
pm_suspicion = pd.read_csv(pm_path) if os.path.exists(pm_path) else pd.DataFrame()

print(f"Master: {len(master)} rows")

Gemini classifications: 41
Master: 379 rows


## Step 2: Oscillation Score (weight: 0.20)

Detect threat-then-reversal patterns within a 72-hour window. Escalation followed by de-escalation (or vice versa) scores higher when accompanied by large price moves.

In [4]:
def calc_oscillation_score(master):
    """Detect threat -> reversal patterns within 72h window."""
    scores = pd.Series(0.0, index=master.index)

    for i in range(len(master)):
        if master.iloc[i]['iran_posts'] == 0:
            continue

        direction = master.iloc[i]['post_direction']
        if direction == 'none' or direction == 'neutral':
            continue

        # Look at next 3 trading days for reversal
        for j in range(1, min(4, len(master) - i)):
            next_dir = master.iloc[i + j]['post_direction']
            if (direction == 'escalation' and next_dir == 'de-escalation') or \
               (direction == 'de-escalation' and next_dir == 'escalation'):
                return_i = abs(master.iloc[i]['daily_return']) if not pd.isna(master.iloc[i]['daily_return']) else 0
                return_j = abs(master.iloc[i + j]['daily_return']) if not pd.isna(master.iloc[i + j]['daily_return']) else 0
                magnitude = (return_i + return_j) / 2

                # Scale: 2% return = 50 score, 4%+ = 100
                osc_score = min(100, magnitude * 25)
                scores.iloc[i] = max(scores.iloc[i], osc_score)
                scores.iloc[i + j] = max(scores.iloc[i + j], osc_score * 0.8)
                break

    return scores

master['oscillation_score'] = calc_oscillation_score(master)
print(f"Days with oscillation score > 0: {(master['oscillation_score'] > 0).sum()}")
print(f"Days with high oscillation (>50): {(master['oscillation_score'] > 50).sum()}")
print(f"Max oscillation score: {master['oscillation_score'].max():.1f}")

Days with oscillation score > 0: 70
Days with high oscillation (>50): 9
Max oscillation score: 100.0


## Step 3: Fabrication Score (weight: 0.25)

Score based on known fabrication events (e.g., Iran denying peace talks) and Gemini fabrication_risk scores. Boosted when a fabricated claim coincided with a large price move.

In [11]:
def calc_fabrication_score(master, gemini):
    """Score based on fabrication flags and Gemini fabrication_risk."""
    scores = pd.Series(0.0, index=master.index)

    # Known fabrication events get high scores
    fabrication_scores = {
        '2026-03-23': 100,  # Iran denied that peace talks occured
        '2026-03-09': 80,   # Contradictory same-day posts
        '2026-03-06': 75,   # Unconditional surrender claim from Trump
        '2026-03-24': 60,   # Trump backed away from deadline
    }

    for date_str, score in fabrication_scores.items():
        mask = master['date'] == pd.Timestamp(date_str)
        scores[mask] = score

    # Add Gemini fabrication_risk for non-flagged days
    if not gemini.empty and 'fabrication_risk' in gemini.columns:
        gemini['timestamp'] = pd.to_datetime(gemini['timestamp'], errors='coerce')
        gemini['date'] = gemini['timestamp'].dt.date
        gemini['fabrication_risk'] = pd.to_numeric(gemini['fabrication_risk'], errors='coerce')

        daily_fab = gemini.groupby('date')['fabrication_risk'].max().reset_index()
        daily_fab['date'] = pd.to_datetime(daily_fab['date'])

        for _, row in daily_fab.iterrows():
            mask = master['date'] == row['date']
            if mask.any() and scores[mask].iloc[0] == 0:
                scores[mask] = row['fabrication_risk']

    # Factor in price impact: fabrication + large price move = higher score
    for i in range(len(master)):
        if scores.iloc[i] > 0:
            price_impact = abs(master.iloc[i]['daily_return']) if not pd.isna(master.iloc[i]['daily_return']) else 0
            impact_boost = min(20, price_impact * 5)
            scores.iloc[i] = min(100, scores.iloc[i] + impact_boost)

    return scores

master['fabrication_score'] = calc_fabrication_score(master, gemini)
print(f"Days with fabrication score > 0: {(master['fabrication_score'] > 0).sum()}")

Days with fabrication score > 0: 19


## Step 4: Causality Score (weight: 0.20)

Score based on pre-announcement volume anomaly (z-score), directional consistency between post tone and price movement, and price move magnitude.

In [13]:
def calc_causality_score(master):
    """Score based on pre-announcement volume anomaly and directional consistency."""
    scores = pd.Series(0.0, index=master.index)

    for i in range(len(master)):
        if master.iloc[i]['iran_posts'] == 0:
            continue

        # Volume anomaly (z-score of absolute return)
        vol_z = master.iloc[i]['volume_anomaly_z']
        vol_score = min(50, max(0, vol_z * 20))

        # Directional consistency
        direction = master.iloc[i]['post_direction']
        daily_ret = master.iloc[i]['daily_return']

        if pd.isna(daily_ret):
            dir_score = 0
        elif direction == 'escalation' and daily_ret > 0:
            dir_score = min(30, daily_ret * 10)
        elif direction == 'de-escalation' and daily_ret < 0:
            dir_score = min(30, abs(daily_ret) * 10)
        else:
            dir_score = 0

        # Price move magnitude
        magnitude = abs(daily_ret) if not pd.isna(daily_ret) else 0
        mag_score = min(20, magnitude * 5)

        scores.iloc[i] = vol_score + dir_score + mag_score

    return scores

master['causality_score'] = calc_causality_score(master)
print(f"Days with causality score > 0: {(master['causality_score'] > 0).sum()}")

Days with causality score > 0: 290


## Step 5: Polymarket Score (weight: 0.20)

Use the Polymarket suspicion values already merged into the master dataset from Phase 4.

In [14]:
master['polymarket_score'] = master['polymarket_suspicion'].fillna(0)
print(f"Days with Polymarket score > 0: {(master['polymarket_score'] > 0).sum()}")

Days with Polymarket score > 0: 310


## Step 6: Gemini Intent Score (weight: 0.15)

Combine Gemini's market_mover_probability (60%) and timing_suspicion (40%) into a single intent score per day.

In [8]:
def calc_intent_score(master, gemini):
    """Score based on market_mover_probability x timing_suspicion from Gemini."""
    scores = pd.Series(0.0, index=master.index)

    if gemini.empty:
        return scores

    if 'market_mover_probability' in gemini.columns and 'timing_suspicion' in gemini.columns:
        gemini['timestamp'] = pd.to_datetime(gemini['timestamp'], errors='coerce')
        gemini['date'] = gemini['timestamp'].dt.date
        gemini['market_mover_probability'] = pd.to_numeric(gemini['market_mover_probability'], errors='coerce')
        gemini['timing_suspicion'] = pd.to_numeric(gemini['timing_suspicion'], errors='coerce')

        # Combine: (market_mover * 0.6 + timing * 0.4)
        gemini['intent'] = gemini['market_mover_probability'] * 0.6 + gemini['timing_suspicion'] * 0.4
        daily_intent = gemini.groupby('date')['intent'].max().reset_index()
        daily_intent['date'] = pd.to_datetime(daily_intent['date'])

        for _, row in daily_intent.iterrows():
            mask = master['date'] == row['date']
            if mask.any():
                scores[mask] = row['intent']

    return scores

master['intent_score'] = calc_intent_score(master, gemini)
print(f"Days with intent score > 0: {(master['intent_score'] > 0).sum()}")

Days with intent score > 0: 20


## Step 7: Compute Composite Score and Assign Verdicts

Weighted sum of all five sub-scores. Verdicts: LOW (<50), ELEVATED (50-69), HIGH (70+).

In [9]:
master['composite_score'] = (
    master['oscillation_score']  * 0.20 +
    master['fabrication_score']  * 0.25 +
    master['causality_score']    * 0.20 +
    master['polymarket_score']   * 0.20 +
    master['intent_score']       * 0.15
)

# Verdicts
master['verdict'] = 'LOW'
master.loc[master['composite_score'] >= 50, 'verdict'] = 'ELEVATED'
master.loc[master['composite_score'] >= 70, 'verdict'] = 'HIGH'

print(f"\nVerdict Distribution:")
print(master['verdict'].value_counts().to_string())

print(f"\nTop 10 Most Suspicious Days:")
top = master.nlargest(10, 'composite_score')
for _, row in top.iterrows():
    print(f"  {row['date'].strftime('%Y-%m-%d')} | Score: {row['composite_score']:>5.1f} | {row['verdict']}")
    print(f"    Osc:{row['oscillation_score']:>4.0f} Fab:{row['fabrication_score']:>4.0f} "
          f"Caus:{row['causality_score']:>4.0f} PM:{row['polymarket_score']:>4.0f} "
          f"Int:{row['intent_score']:>4.0f} | Dir: {row['post_direction']}")


Verdict Distribution:
verdict
LOW         377
ELEVATED      2

Top 10 Most Suspicious Days:
  2026-03-09 | Score:  52.6 | ELEVATED
    Osc:   0 Fab: 100 Caus:  58 PM:  80 Int:   0 | Dir: escalation
  2026-03-23 | Score:  50.1 | ELEVATED
    Osc:   0 Fab: 100 Caus:  45 PM:  80 Int:   0 | Dir: escalation
  2026-03-06 | Score:  49.8 | LOW
    Osc:   0 Fab:  95 Caus: 100 PM:  30 Int:   0 | Dir: escalation
  2025-04-04 | Score:  46.0 | LOW
    Osc: 100 Fab:   0 Caus: 100 PM:  30 Int:   0 | Dir: de-escalation
  2025-10-10 | Score:  42.5 | LOW
    Osc:   0 Fab:  40 Caus:  70 PM:  30 Int:  83 | Dir: mixed
  2025-12-12 | Score:  39.0 | LOW
    Osc:   0 Fab:  96 Caus:   1 PM:  30 Int:  58 | Dir: neutral
  2025-04-09 | Score:  37.3 | LOW
    Osc:  82 Fab:   0 Caus:  75 PM:  30 Int:   0 | Dir: escalation
  2025-11-10 | Score:  36.5 | LOW
    Osc:   0 Fab:  73 Caus:  10 PM:  30 Int:  68 | Dir: escalation
  2025-12-08 | Score:  36.1 | LOW
    Osc:   0 Fab:  85 Caus:  34 PM:  30 Int:  14 | Dir: esca

## Step 8: Save Updated Master

Write the master dataset with all composite score columns.

In [10]:
master.to_csv(os.path.join(PROCESSED, 'master.csv'), index=False)
print(f"Saved master.csv with composite scores ({len(master)} rows, {len(master.columns)} columns)")

Saved master.csv with composite scores (379 rows, 31 columns)
